In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from src.utils.paths import load_paths
from src.utils.logging import setup_logger
import sklearn.metrics as skm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, confusion_matrix
from src.eval.plots import plot_roc_curves, plot_pr_curves, plot_confusion_matrices, plot_probability_distributions

paths = load_paths()
logger = setup_logger(level="INFO")

# 1. Load Artifacts from Balanced Bagging Run
logger.info("Loading Balanced Bagging Artifacts...")

bagging_dir = paths.artifacts_dir / "balanced_bagging"
metrics_path = bagging_dir / "metrics.json"
preds_path = bagging_dir / "predictions.csv"

if not metrics_path.exists() or not preds_path.exists():
    logger.error(f"Artifacts not found in {bagging_dir}. Please run Notebook 10 first.")
    raise FileNotFoundError("Balanced Bagging artifacts missing.")

with open(metrics_path, "r") as f:
    metrics = json.load(f)

df_preds = pd.read_csv(preds_path)

# Ensure split column exists
if "split" not in df_preds.columns:
    logger.warning("'split' column missing in predictions.csv. Assuming 'test'.")
    df_preds["split"] = "test"

logger.info(f"Loaded metrics and {len(df_preds)} predictions.")


2026-03-07 15:12:33 | INFO | ai-vpn-firewall | Loading Balanced Bagging Artifacts...
2026-03-07 15:12:33 | INFO | ai-vpn-firewall | Loaded metrics and 19908 predictions.


In [2]:
print("\n" + "="*60)
print("OPERATIONAL ENSEMBLE OPTIMIZATION: WEIGHTS & SESSION POLICIES")
print("="*60)

# Helper to find threshold for a target FPR
def get_threshold_from_metrics(metrics_dict, target_fpr):
    key = f"fpr_{target_fpr}"
    if key in metrics_dict:
        return metrics_dict[key]["threshold"]
    key_str = f"fpr_{str(target_fpr)}"
    if key_str in metrics_dict:
        return metrics_dict[key_str]["threshold"]
    return 0.99

# --- 1. Ensemble Weight Tuning Grid Search ---
print("\n--- [1] Ensemble Weight Tuning Grid Search (on VAL set) ---")

val_preds = df_preds[df_preds["split"] == "val"].copy()
y_val = val_preds["label"].astype(int).values

# Extract family raw probabilities
# Assuming columns: p_xgb_raw, p_lgbm_raw, p_cat_raw
fam_cols = ["p_xgb_raw", "p_lgbm_raw", "p_cat_raw"]
missing_fam = [c for c in fam_cols if c not in val_preds.columns]

if missing_fam:
    print(f"WARNING: Missing family columns {missing_fam}. Skipping weight tuning.")
else:
    X_val_fam = val_preds[fam_cols].values

    # Define weight grid
    # (xgb, lgbm, cat)
    grid = [
        (1.0, 1.0, 1.0), # Equal (Baseline)
        (2.0, 1.0, 1.0), # XGB heavy
        (1.0, 2.0, 1.0), # LGBM heavy
        (1.0, 1.0, 2.0), # Cat heavy
        (3.0, 1.0, 1.0), # XGB very heavy
        (1.0, 3.0, 1.0), # LGBM very heavy
        (1.0, 1.0, 3.0), # Cat very heavy
        (0.0, 1.0, 1.0), # No XGB
        (1.0, 0.0, 1.0), # No LGBM
        (1.0, 1.0, 0.0), # No Cat
    ]

    results_grid = []

    for w in grid:
        # Normalize weights
        w_norm = np.array(w) / np.sum(w)

        # Compute weighted average
        p_weighted = np.dot(X_val_fam, w_norm)

        # Evaluate at strict FPR (0.1%)
        # We need to find the threshold for this specific weighted combo on VAL
        # Sort p_weighted to find threshold
        desc_score_indices = np.argsort(p_weighted)[::-1]
        y_sorted = y_val[desc_score_indices]
        p_sorted = p_weighted[desc_score_indices]

        distinct_value_indices = np.where(np.diff(p_sorted))[0]
        threshold_idxs = np.r_[distinct_value_indices, y_sorted.size - 1]

        tps = np.cumsum(y_sorted)[threshold_idxs]
        fps = np.cumsum(1 - y_sorted)[threshold_idxs]

        tn = (1 - y_val).sum()
        fprs = fps / tn

        # Find index closest to target FPR 0.001
        target_fpr = 0.001
        idx = np.searchsorted(fprs, target_fpr)
        if idx >= len(threshold_idxs): idx = len(threshold_idxs) - 1

        best_thr = p_sorted[threshold_idxs[idx]]
        achieved_fpr = fprs[idx]
        recall_at_fpr = tps[idx] / y_val.sum()

        auc = roc_auc_score(y_val, p_weighted)

        results_grid.append({
            "weights": w,
            "auc": auc,
            "recall_at_0.1_fpr": recall_at_fpr,
            "achieved_fpr": achieved_fpr,
            "threshold": best_thr
        })

    df_grid = pd.DataFrame(results_grid).sort_values("recall_at_0.1_fpr", ascending=False)
    print("\nTop 5 Weight Configurations (by Recall @ 0.1% FPR on Val):")
    print(df_grid.head(5).to_string(index=False, float_format="%.4f"))

    best_weights = df_grid.iloc[0]["weights"]
    best_thr_val = df_grid.iloc[0]["threshold"]
    print(f"\nSelected Best Weights: {best_weights}")

# --- 2. Session-Level Policy Comparison ---
print("\n--- [2] Session-Level Policy Comparison (on TEST set) ---")

if "capture_id" in df_preds.columns and not missing_fam:
    test_preds = df_preds[df_preds["split"] == "test"].copy()

    # Re-compute best weighted probability for test set
    w_norm_best = np.array(best_weights) / np.sum(best_weights)
    X_test_fam = test_preds[fam_cols].values
    test_preds["p_best_weighted"] = np.dot(X_test_fam, w_norm_best)

    # Aggregation
    # We group by capture_id and compute session-level scores using different policies
    # Policy 1: Max Score (Standard)
    # Policy 2: Mean Score
    # Policy 3: 90th Percentile Score (Robust Max)
    # Policy 4: Fraction of flows > Threshold (Voting)

    # For Policy 4, we need a flow-level threshold. Let's use the one tuned on Val for the best weights.
    flow_thr = best_thr_val

    session_agg = test_preds.groupby("capture_id").agg(
        label=("label", "max"),
        score_max=("p_best_weighted", "max"),
        score_mean=("p_best_weighted", "mean"),
        score_p90=("p_best_weighted", lambda x: np.percentile(x, 90)),
        frac_over_thr=("p_best_weighted", lambda x: (x >= flow_thr).mean())
    ).reset_index()

    y_sess = session_agg["label"].astype(int)

    print(f"\nEvaluating Session Policies (N={len(session_agg)} sessions)")

    policies = [
        ("Max Score", "score_max"),
        ("Mean Score", "score_mean"),
        ("P90 Score", "score_p90"),
        ("Frac > Thr", "frac_over_thr")
    ]

    results_sess = []

    for pol_name, col in policies:
        scores = session_agg[col]
        auc = roc_auc_score(y_sess, scores)

        # Find threshold for 0.1% FPR on TEST (just for evaluation comparison)
        # In reality we'd tune this on Val, but here we want to see the *potential* of the policy
        # Let's actually check the FPR at the *flow-derived* threshold for Max/Mean to be realistic?
        # No, Max/Mean scores have different ranges. We should check AUC and Recall@FixedFPR.

        # Let's compute Recall @ 0.001 FPR (Session Level)
        desc_idxs = np.argsort(scores)[::-1]
        y_s = y_sess.values[desc_idxs]

        tn = (1 - y_sess).sum()
        fps = np.cumsum(1 - y_s)
        tps = np.cumsum(y_s)
        fprs = fps / tn

        idx = np.searchsorted(fprs, 0.001)
        if idx >= len(fprs): idx = len(fprs) - 1

        rec_at_fpr = tps[idx] / y_sess.sum()

        results_sess.append({
            "Policy": pol_name,
            "Session AUC": auc,
            "Sess Recall @ 0.1% FPR": rec_at_fpr
        })

    print(pd.DataFrame(results_sess).to_string(index=False, float_format="%.4f"))

    # --- 3. Operational Simulation ---
    print("\n--- [3] Operational Firewall Simulation (Best Policy) ---")
    # Let's pick "Max Score" with "Best Weights" as the candidate system
    # Apply the VAL-tuned threshold to the TEST set

    # Flow level stats
    y_test_flow = test_preds["label"].astype(int)
    y_hat_flow = (test_preds["p_best_weighted"] >= best_thr_val).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test_flow, y_hat_flow).ravel()
    flow_fpr = fp / (tn + fp)
    flow_rec = tp / (tp + fn)

    print(f"Flow Level (Thr={best_thr_val:.4f}):")
    print(f"  Recall: {flow_rec:.4f}")
    print(f"  FPR:    {flow_fpr:.5f} ({fp} false positives)")

    # Session level stats (Policy: Block if Max(Flow) >= Thr)
    # This is equivalent to: if any flow in session is blocked, block session
    y_hat_sess = (session_agg["score_max"] >= best_thr_val).astype(int)

    tn_s, fp_s, fn_s, tp_s = confusion_matrix(y_sess, y_hat_sess).ravel()
    sess_fpr = fp_s / (tn_s + fp_s)
    sess_rec = tp_s / (tp_s + fn_s)

    print(f"Session Level (Block if Max >= {best_thr_val:.4f}):")
    print(f"  Recall: {sess_rec:.4f}")
    print(f"  FPR:    {sess_fpr:.5f} ({fp_s} false positive sessions)")

    if sess_fpr > 0.01:
        print("WARNING: Session FPR is high. 'Max' policy might be too aggressive.")
    else:
        print("SUCCESS: Session FPR is controlled.")

else:
    print("Skipping session analysis (missing capture_id or family probs).")

print("\n" + "="*60)


OPERATIONAL ENSEMBLE OPTIMIZATION: WEIGHTS & SESSION POLICIES

--- [1] Ensemble Weight Tuning Grid Search (on VAL set) ---

Top 5 Weight Configurations (by Recall @ 0.1% FPR on Val):
        weights    auc  recall_at_0.1_fpr  achieved_fpr  threshold
(2.0, 1.0, 1.0) 0.9809             0.8212        0.0016     0.8517
(3.0, 1.0, 1.0) 0.9809             0.8212        0.0016     0.8496
(1.0, 3.0, 1.0) 0.9798             0.8198        0.0016     0.8687
(1.0, 2.0, 1.0) 0.9805             0.8198        0.0016     0.8649
(1.0, 1.0, 1.0) 0.9811             0.8183        0.0016     0.8586

Selected Best Weights: (2.0, 1.0, 1.0)

--- [2] Session-Level Policy Comparison (on TEST set) ---

Evaluating Session Policies (N=47 sessions)
    Policy  Session AUC  Sess Recall @ 0.1% FPR
 Max Score       0.9471                  0.4118
Mean Score       0.9941                  0.8235
 P90 Score       0.9863                  0.7059
Frac > Thr       0.8480                  0.7059

--- [3] Operational Firewall 